# Experiment 11 — Patient-level cluster bootstrap on exp04 paired tests

A patient-level cluster bootstrap re-evaluates every paired
clean-vs-perturbed test in exp04. Clusters are `subject_id`s; patients are
resampled with replacement (B=1000), all studies per resampled patient are
retained intact, and $\Delta$AUC is recomputed per resample. Two-sided
cluster-bootstrap $p$-values and 95% percentile CIs accompany the DeLong
values in the main manuscript, accommodating within-patient dependence in
MIMIC and (to a lesser extent) NIH.


In [ ]:
import os, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

ROOT = Path(os.environ.get('V4_WORK_DIR',
    '/home/saptpurk/embeddings-noise-eliminators/v4_work'))
B = int(os.environ.get('BOOT_B', 1000))
SEED = 42
rng = np.random.default_rng(SEED)

def _load_exp04():
    dfs = []
    for f in sorted(ROOT.glob('v4_exp04_clean_vs_perturbed_*/*_gpu*.parquet')):
        dfs.append(pd.read_parquet(f))
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

summary = _load_exp04()
out_dir = ROOT / 'v4_exp11_clustered_bootstrap'
out_dir.mkdir(parents=True, exist_ok=True)

if summary.empty:
    pd.DataFrame({'status': ['inputs_absent']}).to_parquet(
        out_dir / 'exp11_manifest.parquet', index=False)
else:
    rows = []
    for _, r in summary.iterrows():
        pred_path = ROOT / f"v4_exp04_clean_vs_perturbed_{r['dataset']}" / (
            f"preds_{r['model']}_{r['disease']}_{r['perturbation']}.parquet")
        if not pred_path.exists():
            rows.append({**r.to_dict(), 'cluster_boot_p': np.nan,
                         'cluster_boot_ci_low': np.nan,
                         'cluster_boot_ci_high': np.nan, 'n_patients': np.nan})
            continue
        preds = pd.read_parquet(pred_path)
        req = {'subject_id', 'y', 'p_clean', 'p_perturbed'}
        if not req.issubset(preds.columns):
            warnings.warn(f'{pred_path} missing {req - set(preds.columns)}')
            continue
        patients = preds['subject_id'].unique()
        deltas = np.empty(B)
        for b in range(B):
            pick = rng.choice(patients, size=len(patients), replace=True)
            sub = preds[preds['subject_id'].isin(pick)]
            try:
                a_c = roc_auc_score(sub['y'], sub['p_clean'])
                a_p = roc_auc_score(sub['y'], sub['p_perturbed'])
                deltas[b] = a_p - a_c
            except ValueError:
                deltas[b] = np.nan
        deltas = deltas[~np.isnan(deltas)]
        p = float(2 * min((deltas >= 0).mean(), (deltas <= 0).mean()))
        lo, hi = np.percentile(deltas, [2.5, 97.5])
        rows.append({**r.to_dict(), 'cluster_boot_p': p,
                     'cluster_boot_ci_low': float(lo),
                     'cluster_boot_ci_high': float(hi),
                     'n_patients': int(len(patients))})
    out = pd.DataFrame(rows)
    out_path = out_dir / 'exp11_clustered_bootstrap.parquet'
    out.to_parquet(out_path, index=False)
    print(f'wrote {len(out)} rows -> {out_path}')
